In [1]:
!pip install -q unsloth
!pip install -q datasets transformers accelerate trl peft

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 817.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

In [2]:
!git clone https://github.com/bindhupagadala10/it-rca-capa-framework

Cloning into 'it-rca-capa-framework'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 59 (delta 16), reused 56 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 4.71 MiB | 8.09 MiB/s, done.
Resolving deltas: 100% (16/16), done.


In [3]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
max_seq_length = 3072

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.6.9: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

Unsloth 2026.6.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [6]:
train_dataset = load_dataset(
    "json",
    data_files="/content/it-rca-capa-framework/data/splits/capa_train.jsonl",
    split="train"
)

val_dataset = load_dataset(
    "json",
    data_files="/content/it-rca-capa-framework/data/splits/capa_val.jsonl",
    split="train"
)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [9]:
def formatting_prompts_func(examples):

    texts = []

    for msgs in examples["messages"]:

        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )

        texts.append(text)

    return {
        "text": texts,
    }

train_dataset = train_dataset.map(
    formatting_prompts_func,
    batched=True,
)

val_dataset = val_dataset.map(
    formatting_prompts_func,
    batched=True,
)

Map:   0%|          | 0/1163 [00:00<?, ? examples/s]

Map:   0%|          | 0/145 [00:00<?, ? examples/s]

In [10]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    dataset_text_field="text",

    max_seq_length=max_seq_length,

    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,

        warmup_steps=20,

        num_train_epochs=3,

        learning_rate=2e-4,

        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),

        logging_steps=10,

        eval_strategy="epoch",

        save_strategy="epoch",

        optim="adamw_8bit",

        weight_decay=0.01,

        lr_scheduler_type="cosine",

        seed=42,

        output_dir="mistral_capa_outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1163 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/145 [00:00<?, ? examples/s]

In [13]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,163 | Num Epochs = 3 | Total steps = 438
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss
1,1.441513,1.397804


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in mistral_capa_outputs/checkpoint-146.


PicklingError: Can't pickle <class 'trl.trainer.sft_config.SFTConfig'>: it's not the same object as trl.trainer.sft_config.SFTConfig

In [19]:
model.save_pretrained("emergency_mistral_epoch1")
tokenizer.save_pretrained("emergency_mistral_epoch1")

Unsloth: Restored added_tokens_decoder metadata in emergency_mistral_epoch1/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in emergency_mistral_epoch1.


('emergency_mistral_epoch1/tokenizer_config.json',
 'emergency_mistral_epoch1/chat_template.jinja',
 'emergency_mistral_epoch1/tokenizer.json')

In [20]:
trainer.args.save_strategy = "no"
trainer.control.should_save = False

In [27]:
trainer.train(resume_from_checkpoint=False)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,163 | Num Epochs = 3 | Total steps = 438
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [28]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32768, 4096, padding_idx=770)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj):

In [29]:
from datasets import load_dataset

test_dataset = load_dataset(
    "json",
    data_files="/content/it-rca-capa-framework/data/splits/capa_test.jsonl",
    split="train"
)

print(len(test_dataset))

Generating train split: 0 examples [00:00, ? examples/s]

146


In [30]:
sample = test_dataset[0]

messages = sample["messages"][:-1]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(prompt[:2000])

<s>[INST] Problem Description:
The production directory infrastructure suffered a critical platform failure, causing an immediate LDAP service crash during a massive automated batch synchronization run from an enterprise data warehouse. The automated ETL process initiated an aggressive, multi-threaded database write operation designed to update organizational hierarchy mappings for over 600,000 corporate identities simultaneously. This unthrottled burst of concurrent modify transactions overwhelmed the OpenLDAP write-ahead logging (WAL) system. Due to an operational mismatch between the data warehouse's transmission rate and the directory backend's database commit speed, the slapd transactional log ring buffer experienced an unhandled overflow state. The resulting memory pointer misalignment corrupted the active transaction table index, forcing the slapd daemon to experience a segmentation fault crash, which immediately halted all network authentication and lookup workflows across the 

In [31]:
inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=False,
)

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

prediction = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(prediction)

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Corrective Actions:
- The directory operations team immediately terminated the active data warehouse ETL synchronization streams to stop the transactional overflow. Engineers executed a database recovery sequence using 'slaptool' to sweep and repair the corrupted transaction blocks. To restore service, the team re-imported the system snapshot from a verified 01:00 AM backup using 'slapadd'. The slapd service was successfully restarted, and startup diagnostics confirmed clean database file integrity and complete synchronization availability across slave consumer nodes. To prevent an immediate recurrence, engineers implemented a dynamic configuration restriction reducing maximum concurrent write operations within slapd until a patch is applied. The team verified that single sign-on verification modules validated normal authentication paths, and user login backlogs drained completely. With processing metrics fully restored and database synchronization channels safe, the active incident br

In [32]:
ground_truth = sample["messages"][-1]["content"]

print("GROUND TRUTH:\n")
print(ground_truth)

GROUND TRUTH:

Corrective Actions:
- The response teams isolated the platform failure by instructing the data warehouse team to completely stop the unthrottled data synchronization streams. They purged the corrupted directory datastore files and utilized the 'slapadd' utility to re-import the database environment structure from a clean, verified snapshot backup file. Following the data recovery, engineers restarted the slapd daemon process and validated system response curves under normal production traffic levels to ensure total directory availability across all operational sites.

Preventive Actions:
- To prevent future occurrences of bulk data-driven directory crashes, the data integration group will configure hard-coded batch limits within the ETL framework, restricting updates to a maximum of 10,000 records per micro-batch execution block. Automated alert rules will be integrated into SCOM to monitor directory write transaction queue depths, triggering an immediate processing free

In [33]:
!pip install -q rouge-score bert-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.9 MB/s eta 0:00:00


In [34]:
import json
from rouge_score import rouge_scorer
from bert_score import score

predictions = []
references = []

for i in range(10):

    sample = test_dataset[i]

    messages = sample["messages"][:-1]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=False,
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    pred = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    ref = sample["messages"][-1]["content"]

    predictions.append(pred)
    references.append(ref)

print("Generated:", len(predictions))

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `tran

Generated: 10


In [35]:
scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

r1 = []
r2 = []
rl = []

for pred, ref in zip(predictions, references):

    scores = scorer.score(ref, pred)

    r1.append(scores["rouge1"].fmeasure)
    r2.append(scores["rouge2"].fmeasure)
    rl.append(scores["rougeL"].fmeasure)

print("ROUGE-1:", round(sum(r1)/len(r1),4))
print("ROUGE-2:", round(sum(r2)/len(r2),4))
print("ROUGE-L:", round(sum(rl)/len(rl),4))

ROUGE-1: 0.5173
ROUGE-2: 0.1439
ROUGE-L: 0.2352


In [36]:
P, R, F1 = score(
    predictions,
    references,
    lang="en",
    verbose=True
)

print("BERTScore Precision:", round(P.mean().item(),4))
print("BERTScore Recall   :", round(R.mean().item(),4))
print("BERTScore F1       :", round(F1.mean().item(),4))

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 1.87 seconds, 5.33 sentences/sec
BERTScore Precision: 0.8856
BERTScore Recall   : 0.8742
BERTScore F1       : 0.8798


In [37]:
import pandas as pd

results = {
    "ROUGE-1": round(sum(r1)/len(r1),4),
    "ROUGE-2": round(sum(r2)/len(r2),4),
    "ROUGE-L": round(sum(rl)/len(rl),4),
    "BERTScore_P": round(P.mean().item(),4),
    "BERTScore_R": round(R.mean().item(),4),
    "BERTScore_F1": round(F1.mean().item(),4),
}

pd.DataFrame([results]).to_csv(
    "capa_metrics.csv",
    index=False
)

pd.DataFrame({
    "prediction": predictions,
    "ground_truth": references,
}).to_json(
    "capa_predictions_10.json",
    orient="records",
    indent=2
)

print(results)

{'ROUGE-1': 0.5173, 'ROUGE-2': 0.1439, 'ROUGE-L': 0.2352, 'BERTScore_P': 0.8856, 'BERTScore_R': 0.8742, 'BERTScore_F1': 0.8798}


In [38]:
import pandas as pd

results = {
    "ROUGE-1": 0.5173,
    "ROUGE-2": 0.1439,
    "ROUGE-L": 0.2352,
    "BERTScore_P": 0.8856,
    "BERTScore_R": 0.8742,
    "BERTScore_F1": 0.8798,
}

pd.DataFrame([results]).to_csv(
    "capa_metrics.csv",
    index=False
)

print("Saved capa_metrics.csv")

Saved capa_metrics.csv


In [39]:
import json

with open("capa_predictions_10.json", "w") as f:
    json.dump(
        [
            {
                "prediction": p,
                "ground_truth": r,
            }
            for p, r in zip(predictions, references)
        ],
        f,
        indent=2,
    )

print("Saved capa_predictions_10.json")

Saved capa_predictions_10.json


In [40]:
!zip -r mistral_results.zip \
    capa_metrics.csv \
    capa_predictions_10.json \
    emergency_mistral_epoch1

  adding: capa_metrics.csv (deflated 33%)
  adding: capa_predictions_10.json (deflated 71%)
  adding: emergency_mistral_epoch1/ (stored 0%)
  adding: emergency_mistral_epoch1/adapter_model.safetensors (deflated 7%)
  adding: emergency_mistral_epoch1/tokenizer.model (deflated 61%)
  adding: emergency_mistral_epoch1/tokenizer_config.json (deflated 96%)
  adding: emergency_mistral_epoch1/adapter_config.json (deflated 59%)
  adding: emergency_mistral_epoch1/README.md (deflated 65%)
  adding: emergency_mistral_epoch1/tokenizer.json (deflated 85%)
  adding: emergency_mistral_epoch1/chat_template.jinja (deflated 74%)


In [41]:
from google.colab import files

files.download("mistral_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>